In [0]:
%run ../config/00_config

In [0]:
%run ../utils/00_utils

In [0]:
adls_options = build_adls_options(
    storage_account_name=ADLS_STORAGE_ACCOUNT_NAME,
    client_id=ADLS_CLIENT_ID,
    tenant_id=ADLS_TENANT_ID,
    client_secret=ADLS_CLIENT_SECRET,
)

df_clientes = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .options(**adls_options)
    .load(SOURCE_PATH)
)

In [0]:
total_linhas = df_clientes.count()
total_colunas = len(df_clientes.columns)

print(f"Total de linhas: {total_linhas}")
print(f"Total de colunas: {total_colunas}")

df_clientes.printSchema()

In [0]:
sensitive_columns = ["senha_hash", "email", "nome", "sobrenome"]
safe_columns = [c for c in df_clientes.columns if c not in sensitive_columns]

display(df_clientes.select(safe_columns).limit(10))

In [0]:
dq_metrics = get_basic_dq_metrics(df_clientes)
df_dq_metrics = spark.createDataFrame([dq_metrics])

display(df_dq_metrics)

In [0]:
display(get_null_summary(df_clientes))